In [2]:
import duckdb
import pandas as pd

# Dùng đường dẫn tuyệt đối trực tiếp
DATA_PATH = r"C:\Users\Admin\finance-analytics\data\fraudTrain.csv"

print("Data path:", DATA_PATH)

# Kết nối DuckDB
con = duckdb.connect()
con.execute(f"CREATE VIEW transactions AS SELECT * FROM read_csv_auto('{DATA_PATH}')")

# Query đầu tiên
result = con.execute("""
    SELECT 
        COUNT(*)                                    AS total_transactions,
        SUM(is_fraud)                               AS total_fraud,
        ROUND(AVG(is_fraud) * 100, 2)               AS fraud_rate_pct,
        ROUND(SUM(CASE WHEN is_fraud = 1 
                  THEN amt ELSE 0 END), 2)          AS total_fraud_amount,
        ROUND(AVG(CASE WHEN is_fraud = 1 
                  THEN amt END), 2)                 AS avg_fraud_amount
    FROM transactions
""").df()

print("\n=== TỔNG QUAN ===")
print(result.to_string(index=False))

Data path: C:\Users\Admin\finance-analytics\data\fraudTrain.csv

=== TỔNG QUAN ===
 total_transactions  total_fraud  fraud_rate_pct  total_fraud_amount  avg_fraud_amount
            1296675       7506.0            0.58          3988088.61            531.32


In [3]:
# Fraud theo category — window function như ở công ty
result2 = con.execute("""
    SELECT 
        category,
        COUNT(*)                                        AS total_transactions,
        SUM(is_fraud)                                   AS total_fraud,
        ROUND(AVG(is_fraud) * 100, 2)                   AS fraud_rate_pct,
        ROUND(AVG(CASE WHEN is_fraud = 1 
                  THEN amt END), 2)                     AS avg_fraud_amount,
        ROUND(SUM(is_fraud) * 100.0 / 
              SUM(SUM(is_fraud)) OVER(), 2)             AS pct_of_total_fraud
    FROM transactions
    GROUP BY category
    ORDER BY fraud_rate_pct DESC
""").df()

print("=== FRAUD THEO CATEGORY ===")
print(result2.to_string(index=False))

=== FRAUD THEO CATEGORY ===
      category  total_transactions  total_fraud  fraud_rate_pct  avg_fraud_amount  pct_of_total_fraud
  shopping_net               97543       1713.0            1.76            999.25               22.82
      misc_net               63287        915.0            1.45            797.01               12.19
   grocery_pos              123638       1743.0            1.41            311.99               23.22
  shopping_pos              116672        843.0            0.72            876.92               11.23
 gas_transport              131659        618.0            0.47             12.29                8.23
      misc_pos               79655        250.0            0.31            218.28                3.33
   grocery_net               45452        134.0            0.29             12.16                1.79
        travel               40507        116.0            0.29              9.06                1.55
 entertainment               94014        233.0       

In [4]:
# Phân tích theo giờ + ranking bằng window function
result3 = con.execute("""
    SELECT 
        EXTRACT(hour FROM CAST(trans_date_trans_time AS TIMESTAMP))  AS hour,
        COUNT(*)                                                      AS total_transactions,
        SUM(is_fraud)                                                 AS total_fraud,
        ROUND(AVG(is_fraud) * 100, 2)                                 AS fraud_rate_pct,
        RANK() OVER (ORDER BY SUM(is_fraud) DESC)                     AS fraud_volume_rank,
        RANK() OVER (ORDER BY AVG(is_fraud) DESC)                     AS fraud_rate_rank
    FROM transactions
    GROUP BY hour
    ORDER BY hour
""").df()

print("=== FRAUD THEO GIỜ + RANKING ===")
print(result3.to_string(index=False))

=== FRAUD THEO GIỜ + RANKING ===
 hour  total_transactions  total_fraud  fraud_rate_pct  fraud_volume_rank  fraud_rate_rank
    0               42502        635.0            1.49                  4                4
    1               42869        658.0            1.53                  3                3
    2               42656        625.0            1.47                  5                5
    3               42769        609.0            1.42                  6                6
    4               41863         46.0            0.11                 21               19
    5               42171         60.0            0.14                 17                7
    6               42300         40.0            0.09                 23               24
    7               42203         56.0            0.13                 18                8
    8               42505         49.0            0.12                 19               16
    9               42185         47.0            0.11   

In [5]:
import os

# Tạo thư mục output
OUTPUT_DIR = r"C:\Users\Admin\finance-analytics\reports"

# Query 1: Fraud summary theo category — cho Power BI
df_category = con.execute("""
    SELECT 
        category,
        COUNT(*)                            AS total_transactions,
        SUM(is_fraud)                       AS total_fraud,
        ROUND(AVG(is_fraud) * 100, 2)       AS fraud_rate_pct,
        ROUND(AVG(CASE WHEN is_fraud = 1 
                  THEN amt END), 2)         AS avg_fraud_amount,
        ROUND(SUM(CASE WHEN is_fraud = 1 
                  THEN amt ELSE 0 END), 2)  AS total_fraud_amount
    FROM transactions
    GROUP BY category
    ORDER BY fraud_rate_pct DESC
""").df()

# Query 2: Fraud theo giờ — cho Power BI
df_hourly = con.execute("""
    SELECT 
        EXTRACT(hour FROM CAST(trans_date_trans_time AS TIMESTAMP)) AS hour,
        COUNT(*)                            AS total_transactions,
        SUM(is_fraud)                       AS total_fraud,
        ROUND(AVG(is_fraud) * 100, 2)       AS fraud_rate_pct
    FROM transactions
    GROUP BY hour
    ORDER BY hour
""").df()

# Query 3: Fraud theo state — cho Power BI
df_state = con.execute("""
    SELECT 
        state,
        COUNT(*)                            AS total_transactions,
        SUM(is_fraud)                       AS total_fraud,
        ROUND(AVG(is_fraud) * 100, 2)       AS fraud_rate_pct,
        ROUND(SUM(CASE WHEN is_fraud = 1 
                  THEN amt ELSE 0 END), 2)  AS total_fraud_amount
    FROM transactions
    GROUP BY state
    ORDER BY total_fraud DESC
""").df()

# Query 4: KPI tổng quan — cho Power BI
df_kpi = con.execute("""
    SELECT 
        COUNT(*)                                    AS total_transactions,
        SUM(is_fraud)                               AS total_fraud,
        ROUND(AVG(is_fraud) * 100, 2)               AS fraud_rate_pct,
        ROUND(SUM(CASE WHEN is_fraud = 1 
                  THEN amt ELSE 0 END), 2)          AS total_fraud_amount,
        ROUND(AVG(CASE WHEN is_fraud = 1 
                  THEN amt END), 2)                 AS avg_fraud_amount,
        ROUND(AVG(CASE WHEN is_fraud = 0 
                  THEN amt END), 2)                 AS avg_normal_amount
    FROM transactions
""").df()

# Export ra CSV cho Power BI
df_category.to_csv(f"{OUTPUT_DIR}\\fraud_by_category.csv", index=False)
df_hourly.to_csv(f"{OUTPUT_DIR}\\fraud_by_hour.csv",     index=False)
df_state.to_csv(f"{OUTPUT_DIR}\\fraud_by_state.csv",     index=False)
df_kpi.to_csv(f"{OUTPUT_DIR}\\fraud_kpi.csv",            index=False)

print("✅ Đã export 4 file CSV vào reports/:")
print(f"   fraud_by_category.csv — {df_category.shape}")
print(f"   fraud_by_hour.csv     — {df_hourly.shape}")
print(f"   fraud_by_state.csv    — {df_state.shape}")
print(f"   fraud_kpi.csv         — {df_kpi.shape}")
print("\n✅ Data đã sẵn sàng cho Power BI!")

✅ Đã export 4 file CSV vào reports/:
   fraud_by_category.csv — (14, 6)
   fraud_by_hour.csv     — (24, 4)
   fraud_by_state.csv    — (51, 5)
   fraud_kpi.csv         — (1, 6)

✅ Data đã sẵn sàng cho Power BI!
